# Learning curves and weight distributions

Draws the figures of chapter 5 of the thesis from trained runs:

* **learning curves** (training and validation energy MSE over the epochs) of the
  runs with and without long-range interaction: Fig. 5.1, 5.2 and A.1-A.6;
* **weight distributions** of the model with long range against the one without:
  Fig. 5.3 (p. 55).

It replaces `Neural-Network/testing_notebook.ipynb` (cells 2-5) and
`Neural-Network/weigtht_distribution.py` of commit `9413bfd`. The plotting code lives
in `lrmlp.analysis` (`pip install -e ".[plot]"`), this notebook only lists the runs.
Each figure is written as a PDF; see `docs/methodology.md` for the figure mapping.

Fill in the run directories below (created by `lrmlp train --config configs/<molecule>.toml`,
the run without long range with `--set model.use_long_range=false`); the presets
write them to `runs/` in the repository root. Paths are relative to the directory of
this notebook, which Jupyter uses as working directory.

In [ ]:
from pathlib import Path

#: ``train.run_root`` of the presets, seen from ``notebooks/``.
RUN_ROOT = Path("../runs")
#: Run directories per molecule: "with" and "without" long-range interaction.
#: Placeholders; replace ``...`` by the timestamp suffix (``YYYYmmdd-HHMMSS``) of your runs.
RUN_DIRS = {
    "Paracetamol": {
        "with": RUN_ROOT / "paracetamol-...",
        "without": RUN_ROOT / "paracetamol-...",
    },
    "Aspirin": {
        "with": RUN_ROOT / "aspirin-...",
        "without": RUN_ROOT / "aspirin-...",
    },
    "Ac-Ala3-NHMe": {
        "with": RUN_ROOT / "ac-ala3-nhme-...",
        "without": RUN_ROOT / "ac-ala3-nhme-...",
    },
    "Stachyose": {
        "with": RUN_ROOT / "stachyose-...",
        "without": RUN_ROOT / "stachyose-...",
    },
}
#: Leading epochs left out of the learning curves (they dominate the scale).
#: The thesis removes the first 19 epochs (p. 53): the first plotted epoch is 20
#: (30 for Stachyose) as in the thesis figures. The legacy code counted epochs from 0,
#: lrmlp counts them from 1, hence 19 and 29.
SKIP = {"Stachyose": 29}
DEFAULT_SKIP = 19
#: Output directory of the figures (``runs/`` is not tracked by git).
FIGURE_DIR = RUN_ROOT / "figures"

## Learning curves (Fig. 5.1, 5.2, A.1-A.6)

`plot_training_log` reads `training_log.csv` of a run (current or legacy header) and
writes `<name>_long_range_energy_mse.pdf` resp. `<name>_no_long_range_energy_mse.pdf`;
the title follows `model.use_long_range` of the run.

In [ ]:
from lrmlp.analysis.plots import plot_training_log

for molecule, runs in RUN_DIRS.items():
    for variant, run_dir in runs.items():
        if not run_dir.is_dir():
            print(f"skipped {molecule} ({variant}): {run_dir} does not exist")
            continue
        written = plot_training_log(
            run_dir,
            output_dir=FIGURE_DIR / "learning_curves",
            skip=SKIP.get(molecule, DEFAULT_SKIP),
        )
        print(molecule, variant, *(path.name for path in written))

## Weight distributions (Fig. 5.3)

`plot_weight_distribution` rebuilds both models from their run directories, loads the
best checkpoint (`epoch_a` / `epoch_b` select another one) and draws the histograms of
all weights inside (−1, 1). As the caption of Fig. 5.3 states, every molecule
compares the run with long range against the one without. (For Ac-Ala3-NHMe the
original script compared two epochs of the same long-range run; see open question F6
in `docs/methodology.md`.)

In [ ]:
from lrmlp.analysis.weights import plot_weight_distribution

for molecule, runs in RUN_DIRS.items():
    if not (runs["with"].is_dir() and runs["without"].is_dir()):
        print(f"skipped {molecule}: run directories missing")
        continue
    out = plot_weight_distribution(
        runs["with"],
        runs["without"],
        FIGURE_DIR / "weights" / f"{molecule}_weight_distribution.pdf",
        title=f"Weight Distribution {molecule}",
    )
    print(molecule, out)